# Realized Cash Flow Visualization

This notebook visualizes **realized** (observed) cash flows for loans that were
active as of June 2020. We examine four individual loan outcomes and the
aggregate portfolio.

## Individual loan examples

| Loan | Outcome | Description |
|------|---------|-------------|
| **Prepaid** | `F12Q20301633` | 30-year loan, prepaid 14 months after cutoff |
| **Defaulted** | `F10Q10036965` | 30-year loan, defaulted 4 months after cutoff |
| **Near-matured** | `F10Q20356204` | 15-year loan, 1 month remaining at data end |
| **Censored** | `F17Q10080834` | 30-year loan, still active at data end (261 months remaining) |

> **Note**: No loans in this cohort have fully matured (data ends 2025-06,
> earliest originations are 2010). The near-matured example is a 15-year loan
> at month 179 of 180.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
%matplotlib inline

DATA_DIR = Path('../data/processed')
EXTERNAL_DIR = Path('../data/external')
FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

LGD = 0.25
CUTOFF = pd.Period('2020-06', 'M')

print('Imports complete.')

---

## 1. Load Data and Identify Cohort

In [ ]:
panel_df = pd.read_parquet(DATA_DIR / 'loan_month_panel.parquet')

surv_df = pd.read_parquet(DATA_DIR / 'survival_data_blumenstock.parquet')
orig_info = surv_df[['loan_sequence_number', 'orig_loan_term']].drop_duplicates('loan_sequence_number')

# Loans alive at cutoff
alive_ids = set(panel_df[panel_df['year_month'] == CUTOFF]['loan_sequence_number'].unique())
print(f'Loans alive at {CUTOFF}: {len(alive_ids):,}')

# Full history for alive loans
cohort_panel = panel_df[panel_df['loan_sequence_number'].isin(alive_ids)].copy()
cohort_panel = cohort_panel.merge(orig_info, on='loan_sequence_number', how='left')
cohort_panel['orig_loan_term'] = cohort_panel['orig_loan_term'].fillna(360).astype(int)

print(f'Cohort panel: {len(cohort_panel):,} loan-months')

---

## 2. Compute Realized Cash Flows

Using the observed `bal_repaid` (% of original balance repaid) from the panel,
we recover the actual outstanding balance:
- **UPB** = `orig_upb` × (1 − `bal_repaid` / 100)

For each loan-month we then compute:
- **Interest** = UPB_start × monthly_rate
- **Scheduled principal** = UPB_start − UPB_after (for non-event months)
- **Prepayment** = remaining UPB returned at prepayment month
- **Recovery** = UPB × (1 − LGD) at default month
- **Loss** = UPB × LGD at default month

In [ ]:
# Recover observed UPB from bal_repaid (% of original balance repaid)
cohort_panel = cohort_panel.sort_values(['loan_sequence_number', 'year_month'])
cohort_panel['observed_upb'] = (
    cohort_panel['orig_upb'] * (1.0 - cohort_panel['bal_repaid'].fillna(0.0) / 100.0)
).clip(lower=0.0)

# UPB after this month's payment = observed UPB at this month
cohort_panel['upb_after'] = cohort_panel['observed_upb']

# UPB at start of this month = previous month's observed UPB (orig_upb for first month)
cohort_panel['upb_start'] = (
    cohort_panel.groupby('loan_sequence_number')['observed_upb']
    .shift(1)
    .fillna(cohort_panel['orig_upb'])
    .clip(lower=0.0)
)

monthly_rate = cohort_panel['int_rate'].values / 100.0 / 12.0
upb_start = cohort_panel['upb_start'].values
upb_after = cohort_panel['upb_after'].values

interest_t = upb_start * monthly_rate
principal_t = np.maximum(upb_start - upb_after, 0.0)

# Event flags
is_prepay = ((cohort_panel['event'] == 1) & (cohort_panel['event_code'] == 1)).values
is_default = ((cohort_panel['event'] == 1) & (cohort_panel['event_code'] == 2)).values

# Realized cash flows
cohort_panel['cf_interest'] = np.where(~is_default, interest_t, 0.0)
cohort_panel['cf_principal'] = np.where(~is_default & ~is_prepay, principal_t, 0.0)
cohort_panel['cf_prepay'] = np.where(is_prepay, upb_start, 0.0)
cohort_panel['cf_recovery'] = np.where(is_default, upb_start * (1 - LGD), 0.0)
cohort_panel['cf_loss'] = np.where(is_default, upb_start * LGD, 0.0)
cohort_panel['cf_total'] = (
    cohort_panel['cf_interest'] + cohort_panel['cf_principal']
    + cohort_panel['cf_prepay'] + cohort_panel['cf_recovery']
)

print('Realized cash flows computed (using observed UPB from bal_repaid).')
print(f'Total interest:   ${cohort_panel["cf_interest"].sum()/1e6:,.1f}M')
print(f'Total principal:  ${cohort_panel["cf_principal"].sum()/1e6:,.1f}M')
print(f'Total prepay:     ${cohort_panel["cf_prepay"].sum()/1e6:,.1f}M')
print(f'Total recovery:   ${cohort_panel["cf_recovery"].sum()/1e6:,.1f}M')
print(f'Total loss:       ${cohort_panel["cf_loss"].sum()/1e6:,.1f}M')
print(f'Total CF:         ${cohort_panel["cf_total"].sum()/1e6:,.1f}M')

---

## 3. Individual Loan Cash Flows

In [ ]:
EXAMPLE_LOANS = {
    'Prepaid': 'F12Q20301633',
    'Defaulted': 'F10Q10036965',
    'Near-matured': 'F10Q20356204',
    'Censored': 'F17Q10080834',
}

# Summary table
rows = []
for label, lid in EXAMPLE_LOANS.items():
    loan = cohort_panel[cohort_panel['loan_sequence_number'] == lid]
    first = loan.iloc[0]
    last = loan.iloc[-1]
    cutoff_row = loan[loan['year_month'] == CUTOFF].iloc[0]
    rows.append({
        'Outcome': label,
        'Loan ID': lid,
        'Orig UPB': f"${first['orig_upb']:,.0f}",
        'Rate': f"{first['int_rate']:.3f}%",
        'FICO': int(first['fico_score']),
        'Term': int(first['orig_loan_term']),
        'Age at cutoff': int(cutoff_row['loan_age']),
        'UPB at cutoff': f"${cutoff_row['upb_start']:,.0f}",
        'Last observed': str(last['year_month']),
        'Total CF': f"${loan['cf_total'].sum():,.0f}",
    })

summary_df = pd.DataFrame(rows).set_index('Outcome')
print(summary_df.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = {
    'Interest': '#4e79a7',
    'Sched Principal': '#59a14f',
    'Prepayment': '#f28e2b',
    'Recovery': '#e15759',
}

for ax, (label, lid) in zip(axes.flat, EXAMPLE_LOANS.items()):
    loan = cohort_panel[cohort_panel['loan_sequence_number'] == lid].copy()
    months = loan['year_month'].dt.to_timestamp()

    ax.bar(months, loan['cf_interest'], width=25, color=colors['Interest'],
           label='Interest', alpha=0.85)
    ax.bar(months, loan['cf_principal'], width=25, bottom=loan['cf_interest'],
           color=colors['Sched Principal'], label='Sched Principal', alpha=0.85)

    # Prepayment or recovery as a spike
    if loan['cf_prepay'].sum() > 0:
        bottom = loan['cf_interest'] + loan['cf_principal']
        ax.bar(months, loan['cf_prepay'], width=25, bottom=bottom,
               color=colors['Prepayment'], label='Prepayment', alpha=0.85)
    if loan['cf_recovery'].sum() > 0:
        ax.bar(months, loan['cf_recovery'], width=25,
               color=colors['Recovery'], label='Recovery', alpha=0.85)

    # Mark cutoff
    ax.axvline(CUTOFF.to_timestamp(), color='black', ls='--', lw=1, alpha=0.5, label='Cutoff (2020-06)')

    ax.set_title(f'{label}: {lid}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Cash Flow ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Realized Cash Flows: Individual Loans', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'realized_cf_individual.png', dpi=150, bbox_inches='tight')
plt.show()

### Cumulative Cash Flows and UPB

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (label, lid) in zip(axes.flat, EXAMPLE_LOANS.items()):
    loan = cohort_panel[cohort_panel['loan_sequence_number'] == lid].copy()
    months = loan['year_month'].dt.to_timestamp()

    cumul_cf = loan['cf_total'].cumsum()
    cumul_interest = loan['cf_interest'].cumsum()
    cumul_principal = (loan['cf_principal'] + loan['cf_prepay'] + loan['cf_recovery']).cumsum()

    ax2 = ax.twinx()

    # UPB on secondary axis
    ax2.fill_between(months, 0, loan['upb_start'].values, alpha=0.1, color='gray')
    ax2.plot(months, loan['upb_start'], color='gray', lw=1, ls='--', label='UPB')
    ax2.set_ylabel('UPB ($)', color='gray')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # Cumulative cash flows on primary axis
    ax.plot(months, cumul_cf, 'k-', lw=2, label='Total CF')
    ax.plot(months, cumul_interest, color='#4e79a7', lw=1.5, ls='--', label='Interest')
    ax.plot(months, cumul_principal, color='#59a14f', lw=1.5, ls='--', label='Principal return')

    ax.axvline(CUTOFF.to_timestamp(), color='black', ls='--', lw=1, alpha=0.5)

    ax.set_title(f'{label}: {lid}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Cumulative CF ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(fontsize=7, loc='upper left')
    ax2.legend(fontsize=7, loc='center right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Cumulative Cash Flows and Outstanding Balance', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'realized_cf_cumulative_individual.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 4. Portfolio Cash Flows

Aggregate realized cash flows across all loans alive at June 2020.

In [ ]:
# Aggregate by calendar month
port_cf = cohort_panel.groupby('year_month').agg(
    n_active=('loan_sequence_number', 'nunique'),
    cf_interest=('cf_interest', 'sum'),
    cf_principal=('cf_principal', 'sum'),
    cf_prepay=('cf_prepay', 'sum'),
    cf_recovery=('cf_recovery', 'sum'),
    cf_loss=('cf_loss', 'sum'),
    cf_total=('cf_total', 'sum'),
    total_upb=('upb_start', 'sum'),
    n_prepay=('cf_prepay', lambda x: (x > 0).sum()),
    n_default=('cf_recovery', lambda x: (x > 0).sum()),
).reset_index()

port_cf['cumul_cf'] = port_cf['cf_total'].cumsum()
port_cf['cumul_interest'] = port_cf['cf_interest'].cumsum()
port_cf['cumul_loss'] = port_cf['cf_loss'].cumsum()
port_cf['date'] = port_cf['year_month'].dt.to_timestamp()

print(f'Portfolio months: {len(port_cf)}')
print(f'Active loans range: {port_cf["n_active"].max():,} -> {port_cf["n_active"].min():,}')
print(f'Total portfolio CF: ${port_cf["cf_total"].sum()/1e6:,.1f}M')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Stacked monthly cash flows
ax = axes[0, 0]
ax.stackplot(
    port_cf['date'],
    port_cf['cf_interest'] / 1e6,
    port_cf['cf_principal'] / 1e6,
    port_cf['cf_prepay'] / 1e6,
    port_cf['cf_recovery'] / 1e6,
    labels=['Interest', 'Sched Principal', 'Prepayment', 'Recovery'],
    colors=['#4e79a7', '#59a14f', '#f28e2b', '#e15759'],
    alpha=0.85,
)
ax.axvline(CUTOFF.to_timestamp(), color='black', ls='--', lw=1, alpha=0.5)
ax.set_title('Monthly Cash Flows by Component')
ax.set_ylabel('$M')
ax.legend(fontsize=8, loc='upper right')

# 2. Cumulative cash flows
ax = axes[0, 1]
ax.plot(port_cf['date'], port_cf['cumul_cf'] / 1e6, 'k-', lw=2, label='Total CF')
ax.plot(port_cf['date'], port_cf['cumul_interest'] / 1e6, '--', color='#4e79a7', lw=1.5, label='Interest')
cumul_prin = (port_cf['cf_principal'] + port_cf['cf_prepay'] + port_cf['cf_recovery']).cumsum()
ax.plot(port_cf['date'], cumul_prin / 1e6, '--', color='#59a14f', lw=1.5, label='Principal return')
ax.axvline(CUTOFF.to_timestamp(), color='black', ls='--', lw=1, alpha=0.5)
ax.set_title('Cumulative Cash Flows')
ax.set_ylabel('$M (cumulative)')
ax.legend(fontsize=8)

# 3. Active loans and total UPB
ax = axes[1, 0]
ax.plot(port_cf['date'], port_cf['n_active'], 'k-', lw=2)
ax.set_title('Active Loans')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax2 = ax.twinx()
ax2.plot(port_cf['date'], port_cf['total_upb'] / 1e9, color='steelblue', lw=1.5, ls='--')
ax2.set_ylabel('Total UPB ($B)', color='steelblue')
ax.axvline(CUTOFF.to_timestamp(), color='black', ls='--', lw=1, alpha=0.5)

# 4. Monthly events
ax = axes[1, 1]
ax.bar(port_cf['date'], port_cf['n_prepay'], width=25, color='#f28e2b', alpha=0.8, label='Prepayments')
ax.bar(port_cf['date'], -port_cf['n_default'], width=25, color='#e15759', alpha=0.8, label='Defaults')
ax.axhline(0, color='black', lw=0.5)
ax.axvline(CUTOFF.to_timestamp(), color='black', ls='--', lw=1, alpha=0.5)
ax.set_title('Monthly Prepayments and Defaults')
ax.set_ylabel('Count')
ax.legend(fontsize=8)

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Portfolio Realized Cash Flows (Loans Alive at {CUTOFF})', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'realized_cf_portfolio.png', dpi=150, bbox_inches='tight')
plt.show()

### Post-Cutoff Portfolio Cash Flows

Zooming in on the projection window (post June 2020) — this is the period
that matters for forward-looking model evaluation.

In [ ]:
post = port_cf[port_cf['year_month'] > CUTOFF].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Monthly CF composition
ax = axes[0]
width = 25
ax.bar(post['date'], post['cf_interest'] / 1e6, width=width, color='#4e79a7', label='Interest')
ax.bar(post['date'], post['cf_principal'] / 1e6, width=width,
       bottom=post['cf_interest'] / 1e6, color='#59a14f', label='Sched Principal')
ax.bar(post['date'], post['cf_prepay'] / 1e6, width=width,
       bottom=(post['cf_interest'] + post['cf_principal']) / 1e6,
       color='#f28e2b', label='Prepayment')
ax.bar(post['date'], post['cf_recovery'] / 1e6, width=width,
       bottom=(post['cf_interest'] + post['cf_principal'] + post['cf_prepay']) / 1e6,
       color='#e15759', label='Recovery')
ax.set_title('Monthly Cash Flows (Post-Cutoff)')
ax.set_ylabel('$M')
ax.legend(fontsize=8)

# 2. Cash flow composition as % of total
ax = axes[1]
total = post['cf_total'].replace(0, np.nan)
ax.stackplot(
    post['date'],
    post['cf_interest'] / total * 100,
    post['cf_principal'] / total * 100,
    post['cf_prepay'] / total * 100,
    post['cf_recovery'] / total * 100,
    labels=['Interest', 'Sched Principal', 'Prepayment', 'Recovery'],
    colors=['#4e79a7', '#59a14f', '#f28e2b', '#e15759'],
    alpha=0.85,
)
ax.set_title('Cash Flow Composition (%)')
ax.set_ylabel('% of Total CF')
ax.set_ylim(0, 100)
ax.legend(fontsize=8, loc='center right')

# 3. Cumulative loss
ax = axes[2]
post_cumul_loss = post['cf_loss'].cumsum()
ax.fill_between(post['date'], 0, post_cumul_loss / 1e6, color='#e15759', alpha=0.3)
ax.plot(post['date'], post_cumul_loss / 1e6, color='#e15759', lw=2)
ax.set_title('Cumulative Credit Loss (Post-Cutoff)')
ax.set_ylabel('$M')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'realized_cf_post_cutoff.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
print(f'Post-cutoff summary ({CUTOFF} to {post["year_month"].max()}):')
print(f'  Months:       {len(post)}')
print(f'  Prepayments:  {post["n_prepay"].sum():,}')
print(f'  Defaults:     {post["n_default"].sum():,}')
print(f'  Total CF:     ${post["cf_total"].sum()/1e6:,.1f}M')
print(f'  Total Loss:   ${post["cf_loss"].sum()/1e6:,.1f}M')

---

## 5. Cash Flow Waterfall: Individual Loan Breakdown

For each example loan, show the lifetime cash flow breakdown as a waterfall chart.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, (label, lid) in zip(axes, EXAMPLE_LOANS.items()):
    loan = cohort_panel[cohort_panel['loan_sequence_number'] == lid]

    components = {
        'Interest': loan['cf_interest'].sum(),
        'Sched\nPrincipal': loan['cf_principal'].sum(),
        'Prepay': loan['cf_prepay'].sum(),
        'Recovery': loan['cf_recovery'].sum(),
    }
    # Remove zero components
    components = {k: v for k, v in components.items() if v > 0}

    comp_colors = {
        'Interest': '#4e79a7',
        'Sched\nPrincipal': '#59a14f',
        'Prepay': '#f28e2b',
        'Recovery': '#e15759',
    }

    names = list(components.keys()) + ['Total']
    values = list(components.values())
    total = sum(values)

    # Stacked bar (waterfall)
    bottom = 0
    for name, val in components.items():
        ax.bar('Total', val, bottom=bottom, color=comp_colors[name], alpha=0.85,
               label=name, width=0.6)
        bottom += val

    # Individual bars
    for i, (name, val) in enumerate(components.items()):
        ax.bar(name, val, color=comp_colors[name], alpha=0.85, width=0.6)

    ax.set_title(f'{label}', fontsize=11, fontweight='bold')
    ax.set_ylabel('$')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.tick_params(axis='x', rotation=0, labelsize=8)
    ax.grid(True, alpha=0.3, axis='y')

    # Add loss annotation for defaulted loan
    loss = loan['cf_loss'].sum()
    if loss > 0:
        ax.annotate(f'Loss: ${loss:,.0f}', xy=(0.5, 0.95), xycoords='axes fraction',
                    ha='center', fontsize=9, color='#e15759', fontweight='bold')

plt.suptitle('Lifetime Cash Flow Breakdown by Component', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'realized_cf_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()